# Notebook 02 — Descriptive Analysis and Distribution Fitting

**DS603 Group Project** · Tan Kok Yong (01426300)

This notebook:
1. Plots daily returns and highlights sub-periods.
2. Computes summary statistics and tests for normality (Jarque–Bera).
3. Fits a Student-t distribution via MLE — this calibrated DGP feeds the Monte Carlo study.
4. Overlays empirical, Normal, and Student-t densities.
5. Compares observed two-sided tail probabilities against the **Chebyshev inequality** bound and the Normal prediction.


In [ ]:
# Install required packages (run once per Colab session)
!pip install -q wrds pandas numpy scipy statsmodels matplotlib seaborn pyarrow

import sys, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import json

from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT = '/content/drive/MyDrive/DS603_Project'
sys.path.insert(0, PROJECT_ROOT)

from src.estimators import fit_student_t

PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
FIG_DIR       = os.path.join(PROJECT_ROOT, 'outputs', 'figures')
TAB_DIR       = os.path.join(PROJECT_ROOT, 'outputs', 'tables')
MDL_DIR       = os.path.join(PROJECT_ROOT, 'models')
for d in (FIG_DIR, TAB_DIR, MDL_DIR):
    os.makedirs(d, exist_ok=True)

sns.set_theme(style='whitegrid', context='notebook')
COLORS = {'full': 'steelblue', 'calm': 'seagreen', 'stressed': 'crimson'}
TICKERS = ['JPM', 'AAPL', 'TSLA']

In [ ]:
df = pd.read_parquet(os.path.join(PROCESSED_DIR, 'returns_clean.parquet'))
df['date'] = pd.to_datetime(df['date'])
print(f'Loaded {len(df):,} rows. Tickers: {df.ticker.unique().tolist()}')
df.head(3)

## 1. Summary statistics

In [ ]:
from scipy.stats import jarque_bera, skew, kurtosis

rows = []
for ticker in TICKERS:
    for period in ['full', 'calm', 'stressed']:
        if period == 'full':
            sub = df[df['ticker'] == ticker]['ret'].dropna()
        else:
            sub = df[(df['ticker'] == ticker) & (df['period'] == period)]['ret'].dropna()
        if len(sub) == 0:
            continue
        jb_stat, jb_p = jarque_bera(sub)
        rows.append({
            'Ticker': ticker,
            'Period': period,
            'N': len(sub),
            'Mean (%)': round(sub.mean() * 100, 4),
            'SD (%)': round(sub.std(ddof=1) * 100, 4),
            'Min (%)': round(sub.min() * 100, 2),
            'Max (%)': round(sub.max() * 100, 2),
            'Skewness': round(skew(sub), 3),
            'Excess Kurtosis': round(kurtosis(sub, fisher=True), 3),
            'JB stat': round(jb_stat, 2),
            'JB p-value': round(jb_p, 4),
        })

stats_table = pd.DataFrame(rows)
stats_table.to_csv(os.path.join(TAB_DIR, '02_summary_stats.csv'), index=False)
stats_table

**Interpretation:**  
- All tickers show excess kurtosis > 3 (Normal has excess kurtosis = 0), consistent with heavy tails.  
- Jarque–Bera tests reject normality for all tickers and sub-periods (p < 0.05).  
- Skewness is negative for most, indicating the left tail (large losses) is heavier than implied by symmetry.  
- This motivates fitting a Student-t distribution as a calibrated heavy-tailed DGP.

## 2. Histograms and Q-Q plots vs. Normal

In [ ]:
fig, axes = plt.subplots(len(TICKERS), 2, figsize=(12, 10))

for i, ticker in enumerate(TICKERS):
    ret = df[df['ticker'] == ticker]['ret'].dropna().values
    loss = -ret

    # Histogram with Normal overlay
    ax = axes[i, 0]
    ax.hist(ret, bins=80, density=True, color='steelblue', alpha=0.6, label='Empirical')
    x = np.linspace(ret.min(), ret.max(), 300)
    ax.plot(x, stats.norm.pdf(x, ret.mean(), ret.std(ddof=1)), 'r-', lw=1.5, label='Normal fit')
    ax.set_title(f'{ticker} — Return distribution')
    ax.set_xlabel('Daily return')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

    # Q-Q plot vs Normal
    ax = axes[i, 1]
    (osm, osr), (slope, intercept, r) = stats.probplot(ret, dist='norm')
    ax.scatter(osm, osr, s=4, alpha=0.4, color='steelblue')
    ax.plot([osm[0], osm[-1]], [slope * osm[0] + intercept, slope * osm[-1] + intercept],
            'r-', lw=1.5, label='Normal line')
    ax.set_title(f'{ticker} — Q-Q plot vs Normal')
    ax.set_xlabel('Theoretical quantiles')
    ax.set_ylabel('Sample quantiles')
    ax.legend(fontsize=8)

plt.suptitle('Return distributions: histograms and Q-Q plots', y=1.01, fontsize=13)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, '02_hist_qq.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Heavy tails show up as deviations from the red line in the Q-Q plot corners.')

## 3. Student-t MLE fit

We fit a three-parameter Student-t (location, scale, degrees of freedom) by MLE using `scipy.stats.t.fit`.  
The fitted parameters become the calibrated DGP for the Monte Carlo study in Notebook 04.

In [ ]:
t_params_all = {}

fit_rows = []
for ticker in TICKERS:
    for period in ['full', 'calm', 'stressed']:
        if period == 'full':
            losses = df[df['ticker'] == ticker]['loss'].dropna().values
        else:
            losses = df[(df['ticker'] == ticker) & (df['period'] == period)]['loss'].dropna().values
        if len(losses) < 30:
            continue
        df_t, loc, scale = fit_student_t(losses)
        t_params_all[f'{ticker}_{period}'] = {'df': df_t, 'loc': loc, 'scale': scale, 'n': len(losses)}
        fit_rows.append({
            'Ticker': ticker, 'Period': period, 'N': len(losses),
            'df (nu)': round(df_t, 3), 'loc': round(loc, 6), 'scale': round(scale, 6)
        })

fit_table = pd.DataFrame(fit_rows)
fit_table.to_csv(os.path.join(TAB_DIR, '02_t_fit_params.csv'), index=False)
print(fit_table.to_string(index=False))

# Save fitted parameters for later notebooks
with open(os.path.join(MDL_DIR, 't_params.json'), 'w') as f:
    json.dump(t_params_all, f, indent=2)
print('\nFitted parameters saved to models/t_params.json')

**Interpretation of degrees of freedom (ν):**  
- ν < 5: very heavy tails (common for individual equities).  
- ν → ∞: Student-t approaches Normal.  
- Low ν in stressed periods confirms heavier tails under market stress.

## 4. Density overlays: Empirical vs. Normal vs. Student-t

In [ ]:
fig, axes = plt.subplots(1, len(TICKERS), figsize=(15, 5))

for ax, ticker in zip(axes, TICKERS):
    losses = df[df['ticker'] == ticker]['loss'].dropna().values
    p = t_params_all[f'{ticker}_full']

    # Kernel density of empirical losses
    kde = stats.gaussian_kde(losses)
    x = np.linspace(losses.min(), losses.max(), 400)

    ax.plot(x, kde(x), 'steelblue', lw=2, label='Empirical (KDE)')
    ax.plot(x, stats.norm.pdf(x, -losses.mean(), losses.std(ddof=1)),
            'r--', lw=1.5, label='Normal fit')
    ax.plot(x, stats.t.pdf(x, df=p['df'], loc=p['loc'], scale=p['scale']),
            'g-', lw=1.5, label=f'Student-t (ν={p["df"]:.1f})')

    ax.set_xlim(np.percentile(losses, 0.5), np.percentile(losses, 99.5))
    ax.set_title(ticker)
    ax.set_xlabel('Daily loss L = −RET')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('Loss density: Empirical vs. Normal vs. Student-t', fontsize=13)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, '02_density_overlay.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. Chebyshev inequality comparison

**Course method :** Chebyshev's inequality gives a distribution-free upper bound on the probability of a deviation from the mean:

$$P(|R - \mu| \geq k\sigma) \leq \frac{1}{k^2}$$

We compare this bound against:
- The **Normal prediction**: $2\Phi(-k)$
- The **observed empirical frequency** from our data

**Why Chebyshev, not Markov?**  
Markov's inequality requires a non-negative random variable. The daily return R (and loss L = −R) can be negative, violating this requirement. Chebyshev applies to any random variable with finite variance and works with deviations from the mean in both directions.

In [ ]:
k_values = np.array([1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0])

cheb_rows = []

fig, axes = plt.subplots(1, len(TICKERS), figsize=(15, 5))

for ax, ticker in zip(axes, TICKERS):
    ret = df[df['ticker'] == ticker]['ret'].dropna().values
    mu, sigma = ret.mean(), ret.std(ddof=1)

    chebyshev_bound   = 1.0 / k_values**2
    normal_prediction = 2 * stats.norm.sf(k_values)
    observed = np.array([
        np.mean(np.abs(ret - mu) >= k * sigma) for k in k_values
    ])

    ax.plot(k_values, chebyshev_bound, 's--', color='darkorange', label='Chebyshev bound')
    ax.plot(k_values, normal_prediction, 'r-o', label='Normal prediction')
    ax.plot(k_values, observed, 'b-^', label='Observed empirical')
    ax.set_yscale('log')
    ax.set_xlabel('k (standard deviations)')
    ax.set_ylabel('P(|R−μ| ≥ kσ)')
    ax.set_title(ticker)
    ax.legend(fontsize=8)

    for k, cb, np_, ob in zip(k_values, chebyshev_bound, normal_prediction, observed):
        cheb_rows.append({
            'Ticker': ticker, 'k': k,
            'Chebyshev_bound': round(cb, 6),
            'Normal_prediction': round(np_, 6),
            'Observed': round(ob, 6),
        })

plt.suptitle('Two-sided tail probability: Chebyshev bound vs. Normal vs. Observed', fontsize=12)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, '02_chebyshev.png'), dpi=150, bbox_inches='tight')
plt.show()

cheb_table = pd.DataFrame(cheb_rows)
cheb_table.to_csv(os.path.join(TAB_DIR, '02_chebyshev_comparison.csv'), index=False)
print('Chebyshev comparison saved.')

**Interpretation:**  
- The **Chebyshev bound** is always valid but very conservative — it always lies above the observed probability.  
- The **Normal prediction** underestimates the actual tail probability at k ≥ 2 for all equities, because real returns are heavier-tailed than Normal.  
- The **observed empirical frequency** sits between Chebyshev (conservative) and Normal (optimistic) at moderate k, but exceeds the Normal prediction significantly at k ≥ 3.  
- This shows that using a Normal VaR/CVaR model at the 99% level (≈ k ≈ 2.33σ threshold) will materially understate risk for heavy-tailed equity returns — the central motivation for comparing estimators.